In [42]:
p = gpd.read_file('PEG/propuesta_v1.shp')
p['geometry'] = p.intersection(masas.dissolve().iloc[0].geometry) 

In [44]:
p.to_file('PEG/propuesta_v1.gpkg')

In [52]:
import xml.etree.ElementTree as ET
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# 1. Diccionario de causas generales (idcausa) extraído del manual oficial
DICCIONARIO_CAUSAS = {
    100: "Rayo / Causa Natural",
    210: "Quemas agrícolas (Rastrojos/Restos)",
    212: "Quema de restos de jardinería (Urbanizaciones)",
    220: "Quemas ganaderas (Pastos)",
    230: "Quemas para control de vegetación (Cunetas/Lindes)",
    240: "Trabajos forestales (Eliminación de restos)",
    250: "Hogueras y barbacoas",
    280: "Líneas eléctricas (Genérico)",
    284: "Líneas eléctricas (Contacto con vegetación)",
    286: "Líneas eléctricas (Transformadores de red)",
    290: "Motores y máquinas (Genérico)",
    294: "Motores y máquinas (Escape o avería de vehículo)",
    310: "Ferrocarril",
    320: "Actividades militares (Prácticas de tiro)",
    334: "Pavesas de incendios próximos",
    400: "Incendio Intencionado",
    500: "Causa desconocida / En investigación"
}

# 2. Diccionario de motivaciones específicas (idmotivacion) para incendios intencionados
DICCIONARIO_MOTIVACIONES = {
    400: "Motivación desconocida",
    401: "Prácticas agrícolas tradicional (se dejan arder/pasan al monte)",
    402: "Prácticas ganaderas tradicional (se dejan arder/pasan al monte)",
    403: "Control de animales (daños en cultivos, ganado, etc.)",
    404: "Eliminar vegetación de montes en explotación forestal",
    405: "Mantener libre de vegetación el monte (concepto tradicional del paisaje)",
    411: "Facilitar o favorecer la caza",
    412: "Conflictos cinegéticos",
    421: "Facilitar el ejercicio de la pesca",
    431: "Disensiones o disputas en cuanto a la titularidad de los montes",
    432: "Obtener la modificación del uso del suelo",
    433: "Modificar la linde de la propiedad",
    434: "Eliminar vegetación forestal en lindes",
    441: "Modificar el precio de la madera",
    442: "Obtener un beneficio (salarios, notoriedad, etc.) en su extinción/restauración",
    443: "Forzar la resolución de Consorcios o Convenios",
    444: "Favorecer la producción de productos del monte (setas, espárragos, etc.)",
    451: "Crear malestar y alarma social",
    452: "Animadversión contra repoblaciones forestales",
    453: "Rechazo a la creación o existencia de Espacios Naturales Protegidos",
    461: "Represalia al reducirse las inversiones públicas en los montes",
    462: "Resentimiento contra expropiaciones",
    463: "Represalia por multas impuestas",
    464: "Venganzas",
    471: "Distraer a la Guardia Civil o la Policía",
    472: "Reclamar presencia policial/Guardia Civil o llamar la atención de autoridades",
    481: "Contemplar las labores de extinción",
    482: "Gamberradas",
    483: "Enfermos mentales (pirómanos y otras)",
    484: "Ritos pseudoreligiosos o satánicos",
    499: "Otras motivaciones (conocidas)"
}

def xml_to_geodataframe(xml_file_path):
    # Parsear el archivo XML
    tree = ET.parse(xml_file_path)
    root = tree.getroot()
    
    records = []
    
    # Iterar sobre cada elemento <Pif> del XML
    for pif in root.findall('.//Pif'):
        pif_data = {}
        
        # 1. Identificadores principales de la raíz
        id_pif = pif.find('idpif')
        num_parte = pif.find('numeroparte')
        pif_data['idpif'] = int(id_pif.text) if id_pif is not None and id_pif.text else None
        pif_data['numeroparte'] = int(num_parte.text) if num_parte is not None and num_parte.text else None
        
        # 2. Año (del bloque pif_comun)
        pif_comun = pif.find('pif_comun')
        if pif_comun is not None:
            anio = pif_comun.find('anio')
            pif_data['anio'] = int(anio.text) if anio is not None and anio.text else None
            
        # 3. Datos de localización espacial
        pif_loc = pif.find('pif_localizacion')
        if pif_loc is not None:
            for elem_name in ['idcomunidad', 'idprovincia', 'idmunicipio', 'paraje', 'cuadricula']:
                elem = pif_loc.find(elem_name)
                if elem is not None and elem.text:
                    pif_data[elem_name] = elem.text
            
            lat_elem = pif_loc.find('latitud')
            lon_elem = pif_loc.find('longitud')
            pif_data['latitud'] = float(lat_elem.text) if lat_elem is not None and lat_elem.text else None
            pif_data['longitud'] = float(lon_elem.text) if lon_elem is not None and lon_elem.text else None

        # 4. Fechas de Inicio y Extinción (del bloque pif_tiempos)
        pif_tiempos = pif.find('pif_tiempos')
        if pif_tiempos is not None:
            fecha_ini = pif_tiempos.find('deteccion')
            fecha_ext = pif_tiempos.find('extinguido')
            
            pif_data['fecha_inicio'] = fecha_ini.text if fecha_ini is not None and fecha_ini.text else None
            pif_data['fecha_extincion'] = fecha_ext.text if fecha_ext is not None and fecha_ext.text else None
        else:
            pif_data['fecha_inicio'] = None
            pif_data['fecha_extincion'] = None

        # 5. Causas y Motivaciones (del bloque pif_causa)
        pif_causa = pif.find('pif_causa')
        if pif_causa is not None:
            id_causa = pif_causa.find('idcausa')
            id_motivacion = pif_causa.find('idmotivacion')
            id_causante = pif_causa.find('idcausante')
            
            cod_causa = int(id_causa.text) if id_causa is not None and id_causa.text else None
            cod_motivacion = int(id_motivacion.text) if id_motivacion is not None and id_motivacion.text else None
            
            pif_data['idcausa'] = cod_causa
            pif_data['idmotivacion'] = cod_motivacion
            pif_data['idcausante'] = int(id_causante.text) if id_causante is not None and id_causante.text else None
            
            # Mapear nombres legibles mediante los diccionarios
            pif_data['causa_general'] = DICCIONARIO_CAUSAS.get(cod_causa, "Código no catalogado")
            pif_data['causa_motivacion'] = DICCIONARIO_MOTIVACIONES.get(cod_motivacion, "No aplica / Otra causa")
        else:
            pif_data['idcausa'] = None
            pif_data['idmotivacion'] = None
            pif_data['idcausante'] = None
            pif_data['causa_general'] = "Sin datos de causa"
            pif_data['causa_motivacion'] = "Sin datos de causa"

        # 6. Superficies afectadas totales (del bloque pif_perdidas)
        pif_perdidas = pif.find('pif_perdidas')
        if pif_perdidas is not None:
            sup_arbolada = pif_perdidas.find('superficiearboladatotal')
            sup_no_arbolada = pif_perdidas.find('superficienoarboladatotal')
            pif_data['superficie_arbolada_ha'] = float(sup_arbolada.text) if sup_arbolada is not None and sup_arbolada.text else 0.0
            pif_data['superficie_no_arbolada_ha'] = float(sup_no_arbolada.text) if sup_no_arbolada is not None and sup_no_arbolada.text else 0.0

        if pif_data:
            records.append(pif_data)
            
    # Crear el DataFrame inicial de Pandas
    df = pd.DataFrame(records)
    
    # Convertir las columnas de texto a tipo datetime nativo de Pandas
    df['fecha_inicio'] = pd.to_datetime(df['fecha_inicio'], errors='coerce')
    df['fecha_extincion'] = pd.to_datetime(df['fecha_extincion'], errors='coerce')
    
    # Filtrar aquellos registros que no contengan coordenadas geográficas válidas
    df_valid_geo = df.dropna(subset=['latitud', 'longitud'])
    
    # Generar la columna espacial con Shapely Point
    geometry = [Point(xy) for xy in zip(df_valid_geo['longitud'], df_valid_geo['latitud'])]
    
    # Instanciar el GeoDataFrame espacial (WGS84 -> EPSG:4326)
    gdf = gpd.GeoDataFrame(df_valid_geo, geometry=geometry, crs="EPSG:4326")
    
    return df

# --- Inicialización ---
# gdf_incendios = xml_to_geodataframe("partes_incendio.xml")
# print(gdf_incendios[['numeroparte', 'fecha_inicio', 'fecha_extincion', 'causa_general']])


In [49]:
gdf = xml_to_geodataframe('BRUTOS/HISTORICO/Xml_20260913_223740_1.xml')

In [50]:
gdf.causa_general.unique()

array(['Causa desconocida / En investigación',
       'Líneas eléctricas (Contacto con vegetación)',
       'Incendio Intencionado',
       'Motores y máquinas (Escape o avería de vehículo)',
       'Quemas agrícolas (Rastrojos/Restos)',
       'Líneas eléctricas (Transformadores de red)',
       'Trabajos forestales (Eliminación de restos)',
       'Quema de restos de jardinería (Urbanizaciones)',
       'Motores y máquinas (Genérico)', 'Hogueras y barbacoas',
       'Actividades militares (Prácticas de tiro)',
       'Pavesas de incendios próximos'], dtype=object)

In [51]:
gdf.to_crs(25830).to_file('BRUTOS/HISTORICO/egif.gpkg')

In [55]:
df = xml_to_geodataframe('BRUTOS/HISTORICO/Xml_20260913_223740_1.xml')

In [56]:
df

,idpif,numeroparte,anio,idcomunidad,idprovincia,idmunicipio,cuadricula,latitud,longitud,fecha_inicio,fecha_extincion,idcausa,idmotivacion,idcausante,causa_general,causa_motivacion,superficie_arbolada_ha,superficie_no_arbolada_ha,paraje
0,662203,1985312183,1985,13,31,216,A05,NaN,NaN,1985-09-03 19:00:00,1985-09-03 21:00:00,230,NaN,2,Quemas para control de vegetación (Cunetas/Lin...,No aplica / Otra causa,0.4,0.00,NaN
1,662204,1985312184,1985,13,31,9,A04,NaN,NaN,1985-09-03 11:00:00,1985-09-03 20:00:00,210,NaN,2,Quemas agrícolas (Rastrojos/Restos),No aplica / Otra causa,25.0,15.00,NaN
2,703322,1989310215,1989,13,31,9,A04,NaN,NaN,1989-02-16 18:00:00,1989-02-16 20:00:00,500,NaN,2,Causa desconocida / En investigación,No aplica / Otra causa,0.0,3.00,NaN
3,703410,1989311552,1989,13,31,216,A05,NaN,NaN,1989-05-07 13:20:00,1989-05-07 14:45:00,500,NaN,2,Causa desconocida / En investigación,No aplica / Otra causa,0.0,1.00,NaN
4,703440,1989311640,1989,13,31,9,A04,NaN,NaN,1989-01-20 11:00:00,1989-01-20 13:00:00,220,NaN,1,Quemas ganaderas (Pastos),No aplica / Otra causa,0.0,1.00,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,575267,2017310257,2017,13,31,9,O05,42.625168,-1.353292,2017-07-07 17:12:00,2017-07-08 05:52:00,334,NaN,1,Pavesas de incendios próximos,No aplica / Otra causa,3.0,1.50,Pinar del Paco
88,1164862,2018310023,2018,13,31,216,A05,42.570886,-1.279304,2018-02-26 18:37:00,2018-02-26 20:55:00,400,401.0,2,Incendio Intencionado,Prácticas agrícolas tradicional (se dejan arde...,0.0,2.00,Carretera del Llano
89,1211878,2020310271,2020,13,31,216,A05,42.563481,-1.285172,2020-10-01 12:06:00,2020-10-01 13:05:00,500,NaN,2,Causa desconocida / En investigación,No aplica / Otra causa,0.0,0.50,VADOLUENGO
90,1222003,2021310186,2021,13,31,216,A05,42.569278,-1.278361,2021-06-12 15:17:00,2021-06-12 18:48:00,500,NaN,2,Causa desconocida / En investigación,No aplica / Otra causa,0.0,0.16,Camino las Fontetas


In [59]:
df.keys()

Index(['idpif', 'numeroparte', 'anio', 'idcomunidad', 'idprovincia',
       'idmunicipio', 'cuadricula', 'latitud', 'longitud', 'fecha_inicio',
       'fecha_extincion', 'idcausa', 'idmotivacion', 'idcausante',
       'causa_general', 'causa_motivacion', 'superficie_arbolada_ha',
       'superficie_no_arbolada_ha', 'paraje'],
      dtype='object')

In [60]:
df['superficie_total'] = df.superficie_arbolada_ha + df.superficie_no_arbolada_ha

In [62]:
df_export = df[['fecha_inicio', 'fecha_extincion', 'causa_general', 'causa_motivacion' , 'superficie_arbolada_ha', 'superficie_no_arbolada_ha', 'superficie_total']].copy()

In [68]:
df_export[['causa_general', 'superficie_total']].groupby('causa_general').agg(
    nnumero_incendios = ('superficie_total', 'count'),
    superficie_total = ('superficie_total', 'sum')
).reset_index().sort_values('superficie_total', ascending=False)

,causa_general,nnumero_incendios,superficie_total
13,Quemas agrícolas (Rastrojos/Restos),7,56.05
1,Causa desconocida / En investigación,36,18.42
11,Pavesas de incendios próximos,3,4.56
5,Incendio Intencionado,10,4.00
14,Quemas ganaderas (Pastos),2,1.80
6,Líneas eléctricas (Contacto con vegetación),6,1.30
16,Rayo / Causa Natural,3,1.10
10,Motores y máquinas (Genérico),3,1.09
7,Líneas eléctricas (Genérico),1,1.00
2,Código no catalogado,2,0.41


In [69]:
df_export.to_csv('BRUTOS/HISTORICO/egif_sin_geom.csv')